In [ ]:
# =================================
# 1_feature_extraction.ipynb
# =================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------------------------------
# Dataset for Reconstructed Images
# ---------------------------------
class ReconstructedDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # list classes from folders
        self.classes = sorted([d.name for d in os.scandir(self.root_dir) if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.image_paths = self._get_image_paths()

    def _get_image_paths(self):
        paths = []
        for c in self.classes:
            d = os.path.join(self.root_dir, c)
            for f in os.listdir(d):
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    paths.append((os.path.join(d, f), self.class_to_idx[c]))
        return paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        p, y = self.image_paths[idx]
        im = Image.open(p).convert('L')  # grayscale
        if self.transform:
            im = self.transform(im)
        return im, y

# ---------------------------------
# Data transform for feature extractor
# ---------------------------------
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),  # WRN expects 3 channels
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# ---------------------------------
# Load Pretrained WideResNet50_2 as feature extractor
# ---------------------------------
feat_extractor = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1)
feat_extractor.fc = nn.Identity()  # remove classification head
feat_extractor = feat_extractor.to(device).eval()

# ---------------------------------
# Feature Extraction Function
# ---------------------------------
@torch.inference_mode()
def extract_features(dataloader):
    feats = []
    labs = []
    for x, y in dataloader:
        x = x.to(device)
        f = feat_extractor(x)                       # (B, 2048)
        f = F.adaptive_avg_pool1d(f.unsqueeze(1), 1).squeeze(1)  # ensure shape
        feats.append(f.cpu())
        labs.append(y)
    feats = torch.cat(feats, dim=0)
    labs = torch.cat(labs, dim=0)
    return feats, labs

# ---------------------------------
# Build Template Library
# ---------------------------------
def build_template_library(template_root, samples_per_class=50, batch_size=32):
    ds = ReconstructedDataset(template_root, transform=train_transform)

    # collect indices by class
    idx_by_class = {i: [] for i in range(len(ds.classes))}
    for i, (_, y) in enumerate(ds.image_paths):
        idx_by_class[y].append(i)

    sel_indices = []
    for c in idx_by_class:
        inds = idx_by_class[c][:samples_per_class] if len(idx_by_class[c]) >= samples_per_class else idx_by_class[c]
        sel_indices.extend(inds)

    # construct sub-dataset in-memory
    sub_imgs = [ds.image_paths[i] for i in sel_indices]
    sub_ds_data = [(Image.open(p).convert('L'), y) for p, y in sub_imgs]

    class SimpleMemDS(Dataset):
        def __init__(self, items, transform):
            self.items = items
            self.transform = transform
        def __len__(self): return len(self.items)
        def __getitem__(self, i):
            im, y = self.items[i]
            im = self.transform(im)
            return im, y

    mem_ds = SimpleMemDS(sub_ds_data, train_transform)
    dl = DataLoader(mem_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    feats, labs = extract_features(dl)

    # compute template vector per class
    Tm = []
    for c in range(len(ds.classes)):
        mask = (labs == c)
        fc = feats[mask]
        if fc.shape[0] == 0:
            Tm.append(torch.zeros(feats.shape[1]))
        else:
            med = torch.median(fc, dim=0).values
            Tm.append(med)
    Tm = torch.stack(Tm, dim=0)
    return Tm, ds.classes

# ---------------------------------
# Run Feature Extraction for Template Library
# ---------------------------------
template_root = '/PATH/TO/TEMPLATE_LIB'  # folder with reconstructed labeled images (by class)
Tm, class_names = build_template_library(template_root, samples_per_class=50, batch_size=32)

print("Template library built.")
print("Classes:", class_names)
print("Template tensor shape:", Tm.shape)
